# 2. SupportSense — Document Processing & Retrieval
## Objective
Inspect the TF-IDF retrieval baseline used by SupportSense. The retriever indexes each document's title and text, then ranks passages by cosine similarity to the incoming question.

This notebook is focused on **retrieval quality**: whether the relevant approved document appears in the top results.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from rag_core import SupportKnowledgeBase

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
docs = json.loads((ROOT / "data" / "company_docs.json").read_text(encoding="utf-8"))
questions = json.loads((ROOT / "data" / "evaluation_questions.json").read_text(encoding="utf-8"))
kb = SupportKnowledgeBase(docs)
print(f"Indexed {len(docs)} documents.")

In [ ]:
def inspect_retrieval(question, k=3):
    hits = kb.retrieve(question, k=k)
    return pd.DataFrame([{
        "rank": i + 1, "doc_id": d["doc_id"], "title": d["title"],
        "similarity": round(d["score"], 3), "excerpt": d["text"][:180] + ("…" if len(d["text"]) > 180 else "")
    } for i, d in enumerate(hits)])

sample_question = questions[0]["question"]
print("Question:", sample_question)
display(inspect_retrieval(sample_question))

## Evaluate top-k retrieval against the labelled expected source
`Recall@k` here means the fraction of questions for which the expected document appears among the top-k retrieved documents. It is not the same as answer correctness.

In [ ]:
rows = []
for q in questions:
    hits = kb.retrieve(q["question"], k=3)
    hit_ids = [d["doc_id"] for d in hits]
    rows.append({
        "question_id": q["question_id"],
        "expected_doc": q.get("expected_doc"),
        "top_1_doc": hit_ids[0] if hit_ids else None,
        "hit_at_1": int(q.get("expected_doc") in hit_ids[:1]) if q.get("expected_doc") else 0,
        "hit_at_3": int(q.get("expected_doc") in hit_ids[:3]) if q.get("expected_doc") else 0,
        "retrieved_count": len(hits)
    })
retrieval_results = pd.DataFrame(rows)
display(retrieval_results)
print("Hit@1:", round(retrieval_results["hit_at_1"].mean(), 3))
print("Hit@3:", round(retrieval_results["hit_at_3"].mean(), 3))

## Interpretation and next experiments
- TF-IDF is inexpensive, deterministic, and easy to inspect, making it a useful baseline.
- Keyword overlap can miss paraphrases and semantically related questions.
- Potential improvements include sentence embeddings, hybrid lexical/vector retrieval, metadata filters, reranking, and chunking longer documents.
- The benchmark is tiny; treat each score as a debugging signal, not a reliable production estimate.